In [0]:
# =====================================================
# CONFIGURATION DU PIPELINE D'INGESTION
# =====================================================

CATALOG = "alpenia_dev"
SCHEMA = "bronze"

BASE_PATH = "/Volumes/alpenia_dev/bronze/landing/"

def table_name(source):
    return f"{CATALOG}.{SCHEMA}.brz_{source}"

print("Catalogue :", CATALOG)
print("Schéma :", SCHEMA)
print("Volume :", BASE_PATH)

In [0]:
# =====================================================
# INSPECTION DES FICHIERS SOURCES
# =====================================================

folders = [
    "contracts",
    "clients",
    "pricing",
    "claims",
    "web_events",
    "expertises"
]

for folder in folders:

    print(f"\n========== {folder.upper()} ==========")

    files = dbutils.fs.ls(BASE_PATH + folder + "/")

    for file in files:
        if not file.isDir():
            print(
                file.name,
                "| Taille :", round(file.size / 1024, 2), "Ko"
            )

In [0]:
# Fonction générique d'écriture
# Plutôt que de répéter le même code six fois, nous allons créer une fonction réutilisable.
def write_bronze_table(df, source_name):

    target_table = f"{CATALOG}.{SCHEMA}.brz_{source_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    print(f"Table créée : {CATALOG}.{SCHEMA}.{target_table}")

In [0]:
# Ingestion des contrats CSV

df_contracts = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("delimiter", ";")
    .option("inferSchema", "true")
    .load(BASE_PATH + "contracts/contracts.csv")
)

write_bronze_table(df_contracts, "contracts")

In [0]:
# Ingestion des clients

df_clients = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(BASE_PATH + "clients/clients.csv")
)

write_bronze_table(df_clients, "clients")

In [0]:
# Ingestion de la tarification

df_pricing = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(BASE_PATH + "pricing/pricing.csv")
)

write_bronze_table(df_pricing, "pricing")

In [0]:
# Ingestion des sinistres Parquet

df_claims = (
    spark.read
    .format("parquet")
    .load(BASE_PATH + "claims/*.parquet")
)

write_bronze_table(df_claims, "claims")

In [0]:
# Ingestion des événements web JSON

df_web_events = (
    spark.read
    .format("json")
    .load(BASE_PATH + "web_events/*.json")
)

write_bronze_table(df_web_events, "web_events")

In [0]:
# Ingestion des expertises JSON

df_expertises = (
    spark.read
    .format("json")
    .option("multiLine", "true")
    .load(BASE_PATH + "expertises/expertises.json")
)

write_bronze_table(df_expertises, "expertises")

In [0]:
%sql SHOW TABLES IN alpenia_dev.bronze

In [0]:
%sql
SHOW TABLES IN alpenia_dev.bronze;

In [0]:
%sql
SELECT 'contracts' AS source, COUNT(*) AS total
FROM alpenia_dev.bronze.brz_contracts

UNION ALL

SELECT 'clients', COUNT(*)
FROM alpenia_dev.bronze.brz_clients

UNION ALL

SELECT 'pricing', COUNT(*)
FROM alpenia_dev.bronze.brz_pricing

UNION ALL

SELECT 'claims', COUNT(*)
FROM alpenia_dev.bronze.brz_claims

UNION ALL

SELECT 'web_events', COUNT(*)
FROM alpenia_dev.bronze.brz_web_events

UNION ALL

SELECT 'expertises', COUNT(*)
FROM alpenia_dev.bronze.brz_expertises;

In [0]:
expected_counts = {
    "brz_contracts": 4000,
    "brz_clients": 3000,
    "brz_pricing": 6,
    "brz_claims": 24000,
    "brz_web_events": 90000,
    "brz_expertises": 450
}

for table, expected in expected_counts.items():
    actual = spark.table(
        f"alpenia_dev.bronze.{table}"
    ).count()

    result = "OK" if actual == expected else "ECART"

    print(f"{table}: attendu={expected}, obtenu={actual} -> {result}")